# Classification

It is a supervised machine learning problem where the model learns from labeled data and predicts a category or class for new observations.

## Workflow

**Load → EDA → Preprocessing → Feature Engineering → Train/Test Split → Separate Numerical/Categorical Columns → Create Preprocessing Pipeline → Transform Data → Train Models → Predict → Evaluate**

Load

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/train.csv")

df.head()

EDA

In [ ]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

In [ ]:
df.describe(include="all")

In [ ]:
print("\nData Types:")
print(df.dtypes)

In [ ]:
print("\nUnique Values:")
for col in df.columns:
    print(col, ":", df[col].nunique())

In [ ]:
print("\nSurvival Distribution:")
print(df["Survived"].value_counts())

print("\nSurvival Percentage:")
print(df["Survived"].value_counts(normalize=True) * 100)

### Preprocessing

- Separate target and features
- Remove irrelevant columns
- Separate numerical and categorical columns
- Handle missing values
- Scale numerical features
- Encode categorical features
- Feature engineering when useful

Use a preprocessing pipeline so the same transformations are applied consistently to train and test data.

Separate Target and Features

In [ ]:
X = df.drop(
    ["Survived", "PassengerId", "Name", "Ticket", "Cabin"],
    axis=1
)

y = df["Survived"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

Separate Numerical / Categorical Columns

In [ ]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

print("Numerical Columns:")
num_features = num_cols.tolist()
print(num_features)

print("\nCategorical Columns:")
cat_features = cat_cols.tolist()
print(cat_features)

Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Create Preprocessing Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

Transform Data

In [ ]:
# Logistic Regression, KNN and SVM.

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

### Evaluation

Classification metrics:
- **Accuracy** → higher is better
- **Precision** → higher is better
- **Recall** → higher is better
- **F1** → higher is better

For imbalanced data, do not rely on Accuracy alone.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate(name, y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    )
    recall = recall_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    )
    f1 = f1_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    )

    return {
        "Model": name,
        "Accuracy": float(round(accuracy, 2)),
        "Precision": float(round(precision, 2)),
        "Recall": float(round(recall, 2)),
        "F1": float(round(f1, 2))
    }

# Logistic Regression

**Use:** Simple classification problems and when an interpretable probability-based baseline is useful.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train_processed, y_train)

y_pred_lr = model.predict(X_test_processed)

result_lr = evaluate("Logistic Regression", y_test, y_pred_lr)
result_lr

# K-Nearest Neighbors (KNN)

**Use:** Small/medium datasets where similar observations are expected to have similar classes; scaling is important.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(
    n_neighbors=5
)

model.fit(X_train_processed, y_train)

y_pred_knn = model.predict(X_test_processed)

result_knn = evaluate("KNN", y_test, y_pred_knn)
result_knn

# Decision Tree Classifier

**Use:** Non-linear tabular data and rule-based, interpretable classification; scaling is not required.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model.fit(X_train_processed, y_train)

y_pred_tree = model.predict(X_test_processed)

result_tree = evaluate("Decision Tree", y_test, y_pred_tree)
result_tree

# Random Forest Classifier

**Use:** Strong general-purpose classification for non-linear tabular data; scaling is not required.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_processed, y_train)

y_pred_rf = model.predict(X_test_processed)

result_rf = evaluate("Random Forest", y_test, y_pred_rf)
result_rf

# Support Vector Machine (SVM)

**Use:** Small/medium datasets with complex class boundaries; scaling is important.

In [ ]:
from sklearn.svm import SVC

model = SVC(
    kernel="rbf",
    C=1.0
)

model.fit(X_train_processed, y_train)

y_pred_svm = model.predict(X_test_processed)

result_svm = evaluate("SVM", y_test, y_pred_svm)
result_svm

# Naive Bayes

**Use:** Fast probabilistic classification, especially common for text and high-dimensional sparse features.

In [ ]:
from sklearn.naive_bayes import GaussianNB

model = GaussianNB()

model.fit(X_train_processed, y_train)

y_pred_nb = model.predict(X_test_processed)

result_nb = evaluate("Naive Bayes", y_test, y_pred_nb)
result_nb

# Gradient Boosting Classifier

**Use:** Strong tabular classification where sequential trees can improve previous errors.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model.fit(X_train_processed, y_train)

y_pred_gb = model.predict(X_test_processed)

result_gb = evaluate("Gradient Boosting", y_test, y_pred_gb)
result_gb

# XGBoost Classifier

**Use:** High-performance gradient boosting for many tabular classification problems.

In [ ]:
# pip install xgboost

from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train_processed, y_train)

y_pred_xgb = model.predict(X_test_processed)

result_xgb = evaluate("XGBoost", y_test, y_pred_xgb)
result_xgb

# LightGBM Classifier

**Use:** Fast gradient boosting, especially useful for larger tabular classification datasets.

In [ ]:
# pip install lightgbm

from lightgbm import LGBMClassifier

model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1
)

model.fit(X_train_processed, y_train)

y_pred_lgbm = model.predict(X_test_processed)

result_lgbm = evaluate("LightGBM", y_test, y_pred_lgbm)
result_lgbm

# CatBoost Classifier

**Use:** Gradient boosting that is especially convenient when categorical features are important.

In [ ]:
# pip install catboost

from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.05,
    depth=6,
    verbose=False,
    random_seed=42
)

model.fit(X_train_processed, y_train)

y_pred_cat = model.predict(X_test_processed)

result_cat = evaluate("CatBoost", y_test, y_pred_cat)
result_cat

# Model Comparison

- **Accuracy:** higher is better
- **Precision:** higher is better
- **Recall:** higher is better
- **F1:** higher is better

For imbalanced classification, F1, Recall, Precision, PR-AUC or ROC-AUC may be more informative than Accuracy.

In [ ]:
results = pd.DataFrame([
    result_lr,
    result_knn,
    result_tree,
    result_rf,
    result_svm,
    result_nb,
    result_gb,
    result_xgb,
    result_lgbm,
    result_cat
])

results.sort_values("F1", ascending=False).reset_index(drop=True)

## Confusion Matrix

Use a confusion matrix to see which classes the model gets right and wrong.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_rf
)

plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

comparison = results.sort_values("F1")

plt.figure(figsize=(10, 6))

plt.barh(
    comparison["Model"],
    comparison["F1"]
)

plt.xlabel("F1 Score (higher is better)")
plt.title("Classification Model Comparison")

plt.gca().invert_yaxis()

plt.show()

In [ ]:
import matplotlib.pyplot as plt

comparison = results.sort_values("Accuracy")

plt.figure(figsize=(10, 6))

plt.barh(
    comparison["Model"],
    comparison["Accuracy"]
)

plt.xlabel("Accuracy (higher is better)")
plt.title("Classification Model Comparison")

plt.gca().invert_yaxis()

plt.show()

# Model Improvement

**Baseline → Feature Engineering → Handle Class Imbalance → Compare Models → Select Promising Model → Hyperparameter Tuning → Cross-Validation → Final Model → Final Test Evaluation**

### Feature Engineering
- Create useful features
- Remove irrelevant features
- Encode categorical variables appropriately
- Transform skewed features when justified

### Handle Class Imbalance
Possible methods:
- `class_weight="balanced"`
- Oversampling / undersampling
- SMOTE
- Threshold adjustment

### Hyperparameter Tuning
Common parameters:
- `C`
- `n_neighbors`
- `max_depth`
- `n_estimators`
- `learning_rate`
- `min_samples_split`

### Cross-Validation
Evaluate the model across multiple folds.

### GridSearchCV
Tests every specified hyperparameter combination.

### RandomizedSearchCV
Tests a selected number of combinations and is useful when the search space is large.

In [ ]:
# RandomizedSearchCV for Random Forest

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

params = {
    "n_estimators": [100, 200, 400],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=params,
    n_iter=20,
    cv=5,
    scoring="f1_weighted",
    random_state=42,
    n_jobs=-1
)

search.fit(X_train_processed, y_train)

print("Best parameters:", search.best_params_)
print("Best CV F1:", search.best_score_)

In [ ]:
best_model = search.best_estimator_

y_pred_final = best_model.predict(X_test_processed)

final_result = evaluate(
    "Tuned Random Forest",
    y_test,
    y_pred_final
)

final_result

# Notes

**Logistic Regression** → simple probability-based classification

**KNN** → classify using nearby observations

**Decision Tree** → rule-based non-linear classification

**Random Forest** → many decision trees

**SVM** → maximum-margin classification; scaling important

**Naive Bayes** → fast probabilistic classification; useful for text

**Gradient Boosting** → sequential trees correcting errors

**XGBoost** → optimized gradient boosting

**LightGBM** → fast gradient boosting

**CatBoost** → boosting with strong categorical-data support